# Environmental Sound Classification

This notebook compares several audio feature representations for classifying environmental sounds in the ESC-10 subset of ESC-50.

The experiments use an RBF Support Vector Machine (SVM) and the five predefined dataset folds. The feature sets are evaluated separately and in combination, with a final experiment adding temporal descriptors to the spectral features.

The notebook can run locally or in Google Colab. The dataset is not included in this repository.


## Dataset setup

Download ESC-50 and extract it so that the dataset directory contains the `audio` and `meta` folders. The metadata file should be available at `meta/esc50.csv`.

Set `DATASET_DIR` below to the extracted `ESC-50-master` folder. In Colab, you can mount Google Drive and point this variable to the folder there. No personal Drive path is required by the notebook.


In [ ]:
from pathlib import Path

import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from IPython.display import Audio, display
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Change this path to the location of your extracted ESC-50 dataset.
DATASET_DIR = Path("ESC-50-master")

# All generated files are written here.
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 22050
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 64
N_MFCC = 13


In [ ]:
# If you are using Google Colab and the dataset is stored in Drive,
# uncomment these lines and set DATASET_DIR to your extracted dataset folder.
#
# from google.colab import drive
# drive.mount("/content/drive")
# DATASET_DIR = Path("/content/drive/MyDrive/datasets/ESC-50-master")

AUDIO_DIR = DATASET_DIR / "audio"
METADATA_PATH = DATASET_DIR / "meta" / "esc50.csv"

if not AUDIO_DIR.is_dir() or not METADATA_PATH.is_file():
    raise FileNotFoundError(
        "ESC-50 was not found. Set DATASET_DIR to the extracted "
        "ESC-50-master directory containing 'audio' and 'meta/esc50.csv'."
    )

metadata = pd.read_csv(METADATA_PATH)
esc10 = metadata[
    metadata["esc10"].astype(str).str.strip().str.lower().eq("true")
].copy()

if esc10.empty:
    raise ValueError("No ESC-10 recordings were found in esc50.csv.")

esc10["fold"] = esc10["fold"].astype(int)

missing_files = [
    filename for filename in esc10["filename"]
    if not (AUDIO_DIR / filename).is_file()
]

print(f"ESC-10 recordings in metadata: {len(esc10)}")
print(f"Available audio files: {len(esc10) - len(missing_files)}")
print(f"Missing audio files: {len(missing_files)}")

if missing_files:
    raise FileNotFoundError(
        "Some ESC-10 audio files are missing. Examples: "
        + ", ".join(missing_files[:10])
    )

print(f"Number of classes: {esc10['category'].nunique()}")
print(f"Dataset folds: {sorted(esc10['fold'].unique())}")


## Inspect the recordings

The ESC-10 subset contains 400 recordings across 10 classes. The following cell checks the class distribution and the number of recordings assigned to each fold.


In [ ]:
print("Recordings per class:")
display(esc10["category"].value_counts().sort_index().rename("count").to_frame())

print("Recordings per fold:")
display(esc10["fold"].value_counts().sort_index().rename("count").to_frame())


## Listen to and visualize one recording

The first ESC-10 recording is used here as an example. Its waveform, Log-Mel spectrogram, and MFCC representation are displayed before feature extraction is run on the full subset.


In [ ]:
sample_row = esc10.iloc[0]
sample_path = AUDIO_DIR / sample_row["filename"]

sample_audio, sample_sr = librosa.load(
    sample_path,
    sr=SAMPLE_RATE,
    mono=True
)

print("Filename:", sample_row["filename"])
print("Class:", sample_row["category"])
print("Original fold:", sample_row["fold"])
print("Sample rate:", SAMPLE_RATE)
print("Duration (seconds):", round(len(sample_audio) / SAMPLE_RATE, 2))

display(Audio(sample_audio, rate=SAMPLE_RATE))

plt.figure(figsize=(12, 3))
librosa.display.waveshow(sample_audio, sr=SAMPLE_RATE)
plt.title(f"Waveform: {sample_row['filename']}")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()


In [ ]:
sample_mel_power = librosa.feature.melspectrogram(
    y=sample_audio,
    sr=SAMPLE_RATE,
    n_fft=N_FFT,
    hop_length=HOP_LENGTH,
    n_mels=N_MELS
)
sample_log_mel = librosa.power_to_db(sample_mel_power, ref=np.max)
sample_mfcc = librosa.feature.mfcc(
    S=sample_log_mel,
    n_mfcc=N_MFCC
)

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

mel_image = librosa.display.specshow(
    sample_log_mel,
    sr=SAMPLE_RATE,
    hop_length=HOP_LENGTH,
    x_axis="time",
    y_axis="mel",
    ax=axes[0]
)
axes[0].set_title("Log-Mel spectrogram")
fig.colorbar(mel_image, ax=axes[0], label="Relative energy (dB)")

mfcc_image = librosa.display.specshow(
    sample_mfcc,
    sr=SAMPLE_RATE,
    hop_length=HOP_LENGTH,
    x_axis="time",
    ax=axes[1],
    cmap="coolwarm"
)
axes[1].set_title("MFCC")
axes[1].set_ylabel("Coefficient")
fig.colorbar(mfcc_image, ax=axes[1], label="Coefficient value")

plt.tight_layout()
plt.show()


## Feature extraction

Two spectral representations are extracted from each recording:

- Log-Mel features: the mean and standard deviation of each of the 64 Mel bands, giving 128 values.
- MFCC features: the mean and standard deviation of 13 coefficients, giving 26 values.

Seven temporal descriptors are also calculated. They summarize onset strength, onset rate, onset interval regularity, onset periodicity, the dominant onset period, and relative RMS variation.


In [ ]:
def extract_spectral_features(audio_path):
    audio, sr = librosa.load(
        audio_path,
        sr=SAMPLE_RATE,
        mono=True
    )

    mel_power = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS
    )
    log_mel = librosa.power_to_db(mel_power, ref=np.max)
    mfcc = librosa.feature.mfcc(S=log_mel, n_mfcc=N_MFCC)

    log_mel_vector = np.concatenate([
        log_mel.mean(axis=1),
        log_mel.std(axis=1)
    ])
    mfcc_vector = np.concatenate([
        mfcc.mean(axis=1),
        mfcc.std(axis=1)
    ])

    return log_mel_vector, mfcc_vector


def extract_temporal_features(audio_path):
    audio, sr = librosa.load(
        audio_path,
        sr=SAMPLE_RATE,
        mono=True
    )

    onset_envelope = librosa.onset.onset_strength(
        y=audio,
        sr=sr,
        hop_length=HOP_LENGTH
    )
    onset_frames = librosa.onset.onset_detect(
        onset_envelope=onset_envelope,
        sr=sr,
        hop_length=HOP_LENGTH,
        units="frames"
    )

    intervals = np.diff(onset_frames) * HOP_LENGTH / sr
    if len(intervals) >= 2:
        interval_cv = intervals.std() / (intervals.mean() + 1e-8)
        regularity = 1.0 / (1.0 + interval_cv)
    else:
        regularity = 0.0

    centered_onsets = onset_envelope - onset_envelope.mean()
    max_lag = min(
        len(centered_onsets),
        int(1.5 * sr / HOP_LENGTH) + 1
    )
    autocorrelation = librosa.autocorrelate(
        centered_onsets,
        max_size=max_lag
    )
    min_lag = max(1, int(0.1 * sr / HOP_LENGTH))

    if (
        len(autocorrelation) > min_lag
        and autocorrelation[0] > 1e-10
    ):
        peak_lag = min_lag + int(
            np.argmax(autocorrelation[min_lag:])
        )
        periodicity = max(
            0.0,
            float(autocorrelation[peak_lag] / autocorrelation[0])
        )
        dominant_period = (
            peak_lag * HOP_LENGTH / sr
            if periodicity > 0 else 0.0
        )
    else:
        periodicity = 0.0
        dominant_period = 0.0

    rms = librosa.feature.rms(
        y=audio,
        frame_length=2048,
        hop_length=HOP_LENGTH
    )[0]
    duration = len(audio) / sr

    return np.array([
        onset_envelope.mean(),
        onset_envelope.std(),
        len(onset_frames) / duration,
        regularity,
        periodicity,
        dominant_period,
        rms.std() / (rms.mean() + 1e-8)
    ], dtype=np.float32)


In [ ]:
log_mel_vectors = []
mfcc_vectors = []
temporal_vectors = []
labels = []
fold_ids = []
filenames = []

for index, row in enumerate(esc10.itertuples(index=False), start=1):
    audio_path = AUDIO_DIR / row.filename

    log_mel_vector, mfcc_vector = extract_spectral_features(audio_path)
    temporal_vector = extract_temporal_features(audio_path)

    log_mel_vectors.append(log_mel_vector)
    mfcc_vectors.append(mfcc_vector)
    temporal_vectors.append(temporal_vector)

    labels.append(row.category)
    fold_ids.append(int(row.fold))
    filenames.append(row.filename)

    if index % 50 == 0:
        print(f"Processed {index}/{len(esc10)} recordings")

X_log_mel = np.vstack(log_mel_vectors)
X_mfcc = np.vstack(mfcc_vectors)
X_temporal = np.vstack(temporal_vectors)

y_labels = np.asarray(labels)
fold_ids = np.asarray(fold_ids)
file_names = np.asarray(filenames)

assert X_log_mel.shape == (400, 128)
assert X_mfcc.shape == (400, 26)
assert X_temporal.shape == (400, 7)
assert np.isfinite(X_log_mel).all()
assert np.isfinite(X_mfcc).all()
assert np.isfinite(X_temporal).all()

np.savez_compressed(
    OUTPUT_DIR / "esc10_features.npz",
    X_log_mel=X_log_mel,
    X_mfcc=X_mfcc,
    X_temporal=X_temporal,
    y_labels=y_labels,
    fold_ids=fold_ids,
    file_names=file_names
)

print("Log-Mel feature shape:", X_log_mel.shape)
print("MFCC feature shape:", X_mfcc.shape)
print("Temporal feature shape:", X_temporal.shape)
print("Saved feature arrays to:", OUTPUT_DIR / "esc10_features.npz")


## Five-fold evaluation

The same RBF SVM configuration is used for all feature representations. Each fold is held out once for testing, while the remaining four folds are used for training.

Standardization is fitted only on the training portion of each split. When feature groups are combined, each standardized group is divided by the square root of its dimensionality before concatenation.


In [ ]:
def evaluate_feature_groups(feature_groups, labels, folds):
    # Return out-of-fold predictions and per-fold scores.
    labels = np.asarray(labels)
    folds = np.asarray(folds)

    predictions = np.empty_like(labels)
    fold_scores = []

    for test_fold in sorted(np.unique(folds)):
        train_mask = folds != test_fold
        test_mask = folds == test_fold

        train_parts = []
        test_parts = []

        for features in feature_groups:
            scaler = StandardScaler()
            train_scaled = scaler.fit_transform(features[train_mask])
            test_scaled = scaler.transform(features[test_mask])

            group_scale = np.sqrt(features.shape[1])
            train_parts.append(train_scaled / group_scale)
            test_parts.append(test_scaled / group_scale)

        X_train = np.hstack(train_parts)
        X_test = np.hstack(test_parts)

        model = SVC(
            kernel="rbf",
            C=1.0,
            gamma="scale"
        )
        model.fit(X_train, labels[train_mask])

        fold_predictions = model.predict(X_test)
        predictions[test_mask] = fold_predictions

        fold_scores.append({
            "fold": int(test_fold),
            "accuracy": accuracy_score(
                labels[test_mask],
                fold_predictions
            ),
            "macro_f1": f1_score(
                labels[test_mask],
                fold_predictions,
                average="macro"
            )
        })

    return predictions, pd.DataFrame(fold_scores)


feature_methods = {
    "Log-Mel": [X_log_mel],
    "MFCC": [X_mfcc],
    "Log-Mel + MFCC": [X_log_mel, X_mfcc],
    "Log-Mel + MFCC + temporal": [
        X_log_mel,
        X_mfcc,
        X_temporal
    ],
}

oof_predictions = {}
score_tables = []

for method_name, feature_groups in feature_methods.items():
    print(f"Evaluating {method_name}...")

    method_predictions, method_scores = evaluate_feature_groups(
        feature_groups,
        y_labels,
        fold_ids
    )

    oof_predictions[method_name] = method_predictions
    method_scores.insert(0, "method", method_name)
    score_tables.append(method_scores)

scores_table = pd.concat(score_tables, ignore_index=True)

summary_table = (
    scores_table
    .groupby("method")[["accuracy", "macro_f1"]]
    .mean()
    .reindex(feature_methods.keys())
)

print("Mean scores across the five folds:")
display(summary_table.round(3))


## Classification reports and confusion matrices

The reports below use the out-of-fold predictions, so every recording is evaluated by a model that was not trained on that recording.


In [ ]:
class_names = np.unique(y_labels)

for method_name, predictions in oof_predictions.items():
    print(f"\n{method_name}")
    print(
        classification_report(
            y_labels,
            predictions,
            labels=class_names,
            digits=3,
            zero_division=0
        )
    )


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
axes = axes.ravel()

for ax, (method_name, predictions) in zip(
    axes,
    oof_predictions.items()
):
    ConfusionMatrixDisplay.from_predictions(
        y_labels,
        predictions,
        labels=class_names,
        display_labels=class_names,
        values_format="d",
        xticks_rotation=90,
        colorbar=False,
        ax=ax
    )
    ax.set_title(method_name)

fig.suptitle("ESC-10: out-of-fold confusion matrices")
plt.tight_layout()
fig.savefig(
    OUTPUT_DIR / "esc10_confusion_matrices.png",
    dpi=200,
    bbox_inches="tight"
)
plt.show()


## Inspecting clock-tick predictions

This section checks how many `clock_tick` recordings are classified correctly by each method and lists recordings that the final feature set still misclassifies.


In [ ]:
clock_tick_mask = y_labels == "clock_tick"

print("Correctly classified clock_tick recordings:")
for method_name, predictions in oof_predictions.items():
    correct = np.sum(
        clock_tick_mask & (predictions == y_labels)
    )
    total = np.sum(clock_tick_mask)
    print(f"{method_name}: {correct}/{total}")

final_method = "Log-Mel + MFCC + temporal"
final_predictions = oof_predictions[final_method]

clock_tick_errors = np.flatnonzero(
    clock_tick_mask & (final_predictions != y_labels)
)

print("\nClock-tick recordings misclassified by the final method:")
for index in clock_tick_errors:
    print(
        file_names[index],
        "| predicted as:",
        final_predictions[index],
        "| fold:",
        fold_ids[index]
    )


In [ ]:
# Optional: listen to the first few clock-tick recordings
# misclassified by the final method.

for index in clock_tick_errors[:3]:
    print(
        file_names[index],
        "| predicted as:",
        final_predictions[index]
    )
    display(Audio(
        filename=str(AUDIO_DIR / file_names[index])
    ))


## Save predictions and evaluation scores

The notebook writes the feature arrays, out-of-fold predictions, fold-level scores, and confusion-matrix figure to the `outputs` directory.


In [ ]:
predictions_table = pd.DataFrame({
    "filename": file_names,
    "fold": fold_ids,
    "true_label": y_labels
})

for method_name, predictions in oof_predictions.items():
    predictions_table[method_name] = predictions

predictions_path = OUTPUT_DIR / "esc10_cv_predictions.csv"
scores_path = OUTPUT_DIR / "esc10_cv_fold_scores.csv"

predictions_table.to_csv(predictions_path, index=False)
scores_table.to_csv(scores_path, index=False)

print("Saved predictions:", predictions_path)
print("Saved fold scores:", scores_path)
print("Prediction table shape:", predictions_table.shape)
display(predictions_table.head())


## Notes

The current experiments use a fixed SVM configuration and handcrafted audio features. They do not include hyperparameter optimization, deep learning models, or evaluation on an independent external dataset.

The reported scores are specific to the ESC-10 subset and the five-fold evaluation used here. Results may vary if the feature extraction settings, classifier parameters, or evaluation protocol are changed.
